# VLA Study on Kaggle

Fine-tuning, quantizing and stress-testing SmolVLA on LIBERO. This notebook runs the `vla-study` project end to end on Kaggle.

### One-time setup (5 minutes)
1. **Verify your phone number** in Kaggle account settings. Without it, Kaggle won't give you GPUs or internet.
2. **Notebook settings (right sidebar):** *Accelerator* → **GPU T4 x2**. *Internet* → **On**. Don't use the P100: current PyTorch builds no longer support it.
3. **Hugging Face token:** create a token with *write* access at huggingface.co → Settings → Access Tokens. In Kaggle: *Add-ons → Secrets* → add it with the name `HF_TOKEN`, and tick it for this notebook. All your checkpoints and results are backed up to a **private** Hub repo, so nothing is lost when a session ends.
4. **Get the project in:** either *Add Input → Upload* `vla-study.zip` as a Kaggle Dataset, or push the project to GitHub and set `GITHUB_URL` below.

### How to use it
- Pick what to run in the **CONFIG** cell (`STAGES`), then *Run All*.
- **Short stages** (tests, baseline, paraphrase review): run interactively.
- **Long stages** (training, big evals): click **Save Version → Save & Run All (Commit)**. Kaggle runs the notebook in the background, so you can close your laptop. Check progress under *Your Work → Versions → Logs*.
- Every session starts from a clean machine. The setup cells reinstall everything (~10 min) and restore your progress from the Hub automatically.

**Budget:** Kaggle gives a weekly GPU quota (check *Settings* for your remaining hours) and each session has a 12-hour limit. Training stops itself at `TRAIN_HOURS` and the next run resumes from the last checkpoint.

## CONFIG: the only cell you normally edit

In [ ]:
# ---- What to run this session (in order). Options:
#   "tests", "baseline_smoke", "baseline", "speedtest", "train_A", "train_B",
#   "eval_trained", "quant_list", "quant", "paraphrase_generate", "paraphrase_eval", "results"
STAGES = ["tests", "baseline_smoke"]

# ---- Where the project code comes from
GITHUB_URL = ""                  # e.g. "https://github.com/<you>/vla-study.git"; leave "" to use the uploaded dataset

# ---- Hugging Face backup repo (private; created automatically)
HF_RUNS_REPO = "<your-hf-username>/vla-study-runs"

# ---- Experiment settings. Keep these identical for A and B so the comparison is fair.
SUITE = "libero_object"
N_EP = 10                        # episodes per task in full evals (10 tasks -> 100 episodes)
STEPS = 20000                    # pick after the speed test; 10000 is a fine fallback
BS = 8                           # T4 has 16 GB; raise to 16 only if the speed test shows spare memory
SAVE_FREQ = 1000                 # frequent checkpoints = little lost if a session ends
EVAL_FREQ = 0                    # 0 = no sim eval during training (saves quota; we evaluate afterwards)
TRAIN_HOURS = 10.5               # stop training after this long so there's time to back up (Kaggle limit: 12 h)
EXTRA_TRAIN_ARGS = ""            # e.g. "--dataset.video_backend=pyav" if video decoding errors appear

# ---- Quantization regexes: set after running the "quant_list" stage
VLM_RE = r"\.vlm\."
EXPERT_RE = r"expert"

# ---- Optional LLM paraphrases (needs extra GPU time); "" to skip
LLM = ""                         # e.g. "Qwen/Qwen2.5-1.5B-Instruct"

## 1. GPU check and paths

In [ ]:
import os, subprocess, sys, json, shutil, glob, re, time, signal, threading
from pathlib import Path

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPUs:", gpu or "NONE")
assert gpu, "No GPU. Settings -> Accelerator -> GPU T4 x2 (and verify your phone number)."
if "P100" in gpu:
    raise SystemExit("P100 is not supported by current PyTorch. Switch the accelerator to GPU T4 x2.")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # one GPU per job; see the tip at the end for using both

WORK    = Path("/kaggle/working")
PROJECT = WORK / "vla-study"                       # code (small)
OUT     = WORK / "vla-outputs"                     # checkpoints, evals, results (pruned + backed up)
VENV    = Path("/tmp/venv")                        # Python 3.12 environment (big; NOT saved with the notebook)
LEROBOT = Path("/tmp/lerobot")
os.environ["HF_HOME"] = "/tmp/hf"                  # model/dataset cache, also kept out of /kaggle/working
OUT.mkdir(parents=True, exist_ok=True)
want = lambda stage: stage in STAGES
COMMON = dict(SUITE=SUITE, N_EP=N_EP, STEPS=STEPS, BS=BS, SAVE_FREQ=SAVE_FREQ, EVAL_FREQ=EVAL_FREQ,
              EXTRA_TRAIN_ARGS=EXTRA_TRAIN_ARGS, DEVICE="cuda")   # passed to every script
print("Stages this session:", STAGES)

## 2. Helper for running commands

Streams the important lines, saves the full log to `vla-outputs/logs/`, and can stop a command cleanly after a time limit.

In [ ]:
def run(cmd, log=None, show=None, env=None, hours=None, ok_codes=(0,)):
    # Run a shell command in the project folder.
    #   show : regex - only print matching lines (None = print everything)
    #   log  : file name under vla-outputs/logs/ (full output is always kept there)
    #   hours: send Ctrl-C to the whole process group after this many hours
    e = os.environ.copy(); e.update({k: str(v) for k, v in (env or {}).items()})
    (OUT / "logs").mkdir(exist_ok=True)
    logf = open(OUT / "logs" / log, "a") if log else None
    pat = re.compile(show) if show else None
    p = subprocess.Popen(["bash", "-c", cmd], cwd=PROJECT if PROJECT.exists() else WORK, env=e,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         start_new_session=True)
    timed_out = threading.Event()
    if hours:
        def stop():
            timed_out.set(); print(f">>> time budget of {hours} h reached, stopping cleanly", flush=True)
            os.killpg(p.pid, signal.SIGINT)
        timer = threading.Timer(hours * 3600, stop); timer.daemon = True; timer.start()
    tail = []
    for line in p.stdout:
        if logf: logf.write(line); logf.flush()
        tail = (tail + [line])[-40:]
        if pat is None or pat.search(line): print(line, end="", flush=True)
    code_ = p.wait()
    if logf: logf.close()
    if timed_out.is_set():
        return "timeout"
    if code_ not in ok_codes:
        print("".join(tail))
        raise RuntimeError(f"command failed (exit {code_}): {cmd}")
    return code_

IMPORTANT = r"step|loss|success|pc_success|\[quant\]|\[paraphrase\]|\[sync\]|Error|error|Traceback|WARNING|Warning"

## 3. Get the project code

In [ ]:
if GITHUB_URL:
    shutil.rmtree(PROJECT, ignore_errors=True)
    run(f"git clone -q {GITHUB_URL} {PROJECT}")
else:
    # Kaggle unpacks uploaded zips; accept either a folder or a zip
    hits = [Path(p).parent for p in glob.glob("/kaggle/input/**/vla_study/__init__.py", recursive=True)]
    zips = glob.glob("/kaggle/input/**/vla-study*.zip", recursive=True)
    shutil.rmtree(PROJECT, ignore_errors=True)
    if hits:
        shutil.copytree(hits[0].parent, PROJECT)          # copy: /kaggle/input is read-only
    elif zips:
        run(f"unzip -q '{zips[0]}' -d /tmp/vla_unzip && cp -r /tmp/vla_unzip/vla-study {PROJECT}")
    else:
        raise SystemExit("Project not found. Add vla-study.zip via 'Add Input', or set GITHUB_URL.")

# Point outputs/, data/ and results/ at vla-outputs so everything lands in one backed-up place
for name, target in [("outputs", OUT), ("data", OUT / "data"), ("results", OUT / "results")]:
    target.mkdir(parents=True, exist_ok=True)
    link = PROJECT / name
    if link.is_symlink() or link.is_file(): link.unlink()
    elif link.is_dir():
        for f in link.iterdir():                          # keep any files that shipped with the repo
            if not (target / f.name).exists(): shutil.move(str(f), target / f.name)
        shutil.rmtree(link)
    link.symlink_to(target)
os.chdir(PROJECT)
print(sorted(p.name for p in PROJECT.iterdir()))

## 4. Hugging Face login and restore previous progress

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    raise SystemExit(f"Could not read the HF_TOKEN secret ({e}). Add-ons -> Secrets -> attach HF_TOKEN to this notebook.")
assert "<your-hf-username>" not in HF_RUNS_REPO, "Set HF_RUNS_REPO in the CONFIG cell."

# The sync tool only needs huggingface_hub, which Kaggle's own Python already has
def sync_cmd(action, extra=""):
    return f"{sys.executable} -m vla_study.hub_sync {action} --repo {HF_RUNS_REPO} --outputs {OUT} {extra}"
run(sync_cmd("pull"))
print("Restored files:", sorted(str(p.relative_to(OUT)) for p in OUT.glob("*")))

## 5. Install a Python 3.12 environment with LeRobot (~10 min)

LeRobot needs Python ≥ 3.12 and a recent PyTorch, so we build a separate environment with `uv` instead of using Kaggle's default Python. The LeRobot commit is pinned on the first run and re-used afterwards, so all your runs use the same code.

In [ ]:
run("pip install -q uv")
run("apt-get -qq update && apt-get -qq install -y ffmpeg libegl1 libgl1 libosmesa6 > /dev/null", ok_codes=(0, 100))

PIN = OUT / "LEROBOT_COMMIT.txt"
if not LEROBOT.exists():
    run(f"git clone -q --filter=blob:none https://github.com/huggingface/lerobot.git {LEROBOT}")
if PIN.exists():
    run(f"cd {LEROBOT} && git checkout -q {PIN.read_text().strip()}")
else:
    PIN.write_text(subprocess.check_output(["git", "-C", str(LEROBOT), "rev-parse", "HEAD"], text=True).strip())
print("LeRobot commit:", PIN.read_text().strip())

if not (VENV / "bin" / "python").exists():
    run(f"uv venv -q {VENV} --python 3.12")
run(f"uv pip install -q --python {VENV}/bin/python --torch-backend=auto "
    f"-e '{LEROBOT}[smolvla,libero]' -e {PROJECT} pytest matplotlib", show=r"error|Error")

# From now on, `python`, `lerobot-train`, `lerobot-eval` all mean the 3.12 environment
os.environ["PATH"] = f"{VENV}/bin:" + os.environ["PATH"]
os.environ["VIRTUAL_ENV"] = str(VENV)
# LIBERO may ask a yes/no question on first import; answer it so background runs don't hang
run("printf 'N\\n' | python -c 'import libero.libero' || true")
run("python -c \"import sys, torch, lerobot; print('python', sys.version.split()[0], '| torch', torch.__version__, "
    "'| CUDA', torch.cuda.is_available(), '| bf16', torch.cuda.is_bf16_supported())\"")

## 6. Check the simulator can render

The GPU renderer (EGL) is fastest. If Kaggle's image doesn't support it, we fall back to the CPU renderer (OSMesa), which is slower but works.

In [ ]:
test = ("import mujoco; m = mujoco.MjModel.from_xml_string('<mujoco><worldbody><geom size=\"1\"/></worldbody></mujoco>'); "
        "r = mujoco.Renderer(m, 64, 64); r.update_scene(mujoco.MjData(m)); print(r.render().shape)")
for backend in ("egl", "osmesa"):
    r = subprocess.run(["python", "-c", test], capture_output=True, text=True,
                       env={**os.environ, "MUJOCO_GL": backend, "PYOPENGL_PLATFORM": backend})
    if r.returncode == 0:
        os.environ["MUJOCO_GL"] = os.environ["PYOPENGL_PLATFORM"] = backend
        print(f"Rendering works with {backend}" + ("" if backend == "egl" else " (CPU - evals will be slower)"))
        break
    print(f"{backend} failed: {r.stderr.strip().splitlines()[-1] if r.stderr.strip() else '?'}")
else:
    raise SystemExit("No working renderer; see the troubleshooting notes at the end.")

## 7. Start background backups

Every 15 minutes: delete old checkpoints (keeps the newest two per experiment) and upload new results plus the latest checkpoint to your private Hub repo.

In [ ]:
subprocess.run(["pkill", "-f", "vla_study.hub_sync loop"])        # avoid duplicates if this cell re-runs
(OUT / "logs").mkdir(exist_ok=True)
sync_log = open(OUT / "logs" / "sync.log", "a")
sync_proc = subprocess.Popen(sync_cmd("loop", "--every 900 --keep 2"),
                             shell=True, cwd=PROJECT, stdout=sync_log, stderr=subprocess.STDOUT)
print("backup loop running, pid", sync_proc.pid, "- log: vla-outputs/logs/sync.log")

---
# Stages
Each cell below only runs if its stage is listed in `STAGES`.

### tests: unit tests (should all pass)

In [ ]:
if want("tests"):
    run("python -m pytest tests -q")

### baseline_smoke / baseline: check the evaluation pipeline (do NOT skip)

The smoke test runs 1 episode per task. If the full baseline is near 0% success, stop and debug using Part 4 of the learning guide before training anything.

In [ ]:
from IPython.display import Video, display

def show_video(pattern):
    vids = sorted(glob.glob(str(OUT / "eval" / pattern / "**" / "*.mp4"), recursive=True))
    print(len(vids), "videos")
    if vids: display(Video(vids[0], embed=True, width=400))

if want("baseline_smoke"):
    run("bash scripts/01_eval_baseline.sh", log="baseline_smoke.log", show=IMPORTANT, env={**COMMON, "N_EP": 1})
    show_video("00_baseline_pretrained")
if want("baseline"):
    shutil.rmtree(OUT / "eval" / "00_baseline_pretrained", ignore_errors=True)
    run("bash scripts/01_eval_baseline.sh", log="baseline.log", show=IMPORTANT, env=COMMON)
    show_video("00_baseline_pretrained")

### speedtest: how long will training take?

Trains 200 steps, then deletes the run. Read the seconds-per-step from the log: total hours ≈ sec/step × STEPS ÷ 3600. If 20k steps won't fit your quota, use 10k, and use the same value for A and B.

In [ ]:
if want("speedtest"):
    run("bash scripts/02_train_expert_only.sh", log="speedtest.log", show=IMPORTANT,
        env={**COMMON, "STEPS": 200, "SAVE_FREQ": 1000, "EVAL_FREQ": 0})
    shutil.rmtree(OUT / "train" / "expert_only", ignore_errors=True)
    run("nvidia-smi --query-gpu=memory.used,memory.total --format=csv")

### train_A / train_B: training (run these as **Save & Run All**)

Starts fresh the first time, and automatically resumes from the last checkpoint after that. Stops itself after `TRAIN_HOURS`; then commit again to continue.

In [ ]:
def train(exp, script):
    cfg = OUT / "train" / exp / "checkpoints" / "last" / "pretrained_model" / "train_config.json"
    if cfg.exists():
        print(f"Resuming {exp} from {cfg.parent.parent.resolve().name}")
        cmd = f"lerobot-train --config_path={cfg} --resume=true"
    else:
        if (OUT / "train" / exp).exists():   # a crashed run with no usable checkpoint
            print(f"Removing incomplete run {exp}"); shutil.rmtree(OUT / "train" / exp)
        cmd = f"bash scripts/{script}"
    result = run(cmd, log=f"train_{exp}.log", show=IMPORTANT, env=COMMON, hours=TRAIN_HOURS)
    run(sync_cmd("push"))   # back up right away
    done = (OUT / "train" / exp / "checkpoints" / "last").resolve().name
    print(f"{exp}: latest checkpoint = step {done}" + (" (time budget hit - commit again to continue)" if result == "timeout" else ""))

if want("train_A"):
    train("expert_only", "02_train_expert_only.sh")
if want("train_B"):
    train("expert_plus_vlm", "03_train_expert_plus_vlm.sh")

### eval_trained: closed-loop evaluation of the trained checkpoints

In [ ]:
if want("eval_trained"):
    final = lambda exp: (OUT / "train" / exp / "checkpoints" / "last").resolve().name
    for exp in ("expert_only", "expert_plus_vlm"):
        if (OUT / "train" / exp).exists():
            print(f"{exp}: evaluating step {final(exp)} (should equal STEPS={STEPS:06d} for a finished run)")
    run("bash scripts/04_eval_trained.sh", log="eval_trained.log", show=IMPORTANT, env=COMMON)
    show_video("01_expert_only")

### quant_list / quant: quantization

Run `quant_list` first. Read which module-name prefixes belong to the VLM backbone and which to the action expert, set `VLM_RE` and `EXPERT_RE` in CONFIG so each matches only its part, then run `quant`.

In [ ]:
CKPT_A = "outputs/train/expert_only/checkpoints/last/pretrained_model"
if want("quant_list"):
    run(f"python -m vla_study.quantize --list {CKPT_A}")
if want("quant"):
    run("bash scripts/05_quant_sweep.sh", log="quant.log", show=IMPORTANT,
        env={**COMMON, "VLM_RE": VLM_RE, "EXPERT_RE": EXPERT_RE})
    print(json.dumps(json.load(open(OUT / "offline_bench.json")), indent=1)[:3000])

### paraphrase_generate: build the instruction sets, then REVIEW them

In [ ]:
TASKS = OUT / "data" / f"tasks_{SUITE}.json"
PARA  = OUT / "data" / f"paraphrases_{SUITE}.json"
if want("paraphrase_generate"):
    run(f"python -m vla_study.paraphrase extract --suite {SUITE} --out {TASKS}")
    run(f"python -m vla_study.paraphrase generate --tasks {TASKS} --out {PARA}" + (f" --llm {LLM}" if LLM else ""))
if PARA.exists():
    import pandas as pd
    sets = json.load(open(PARA))
    pd.set_option("display.max_colwidth", 80)
    display(pd.DataFrame({lvl: list(m.values()) for lvl, m in sets.items() if lvl != "original"},
                         index=list(sets["original"])))

**Review every row above.** If a paraphrase changes the meaning (different object, different place), fix it in `FIXES` below and run the cell. Leave `FIXES` empty if everything is fine. The `swapped` column is meant to be wrong: it's the control.

In [ ]:
FIXES = {
    # "level": {"original instruction": "corrected paraphrase"},
    # "llm": {"pick up the alphabet soup and place it in the basket": "put the alphabet soup into the basket"},
}
if PARA.exists() and FIXES:
    sets = json.load(open(PARA))
    for lvl, fixes in FIXES.items():
        for orig, new in fixes.items():
            assert orig in sets[lvl], f"not an original instruction: {orig!r}"
            sets[lvl][orig] = new
    json.dump(sets, open(PARA, "w"), indent=2)
    print("saved fixes; the backup loop will upload them")

### paraphrase_eval: run the robustness evaluation

Check each run's log for `[paraphrase] rewritten instructions: N`. If N is 0, that run did not test anything.

In [ ]:
if want("paraphrase_eval"):
    assert PARA.exists(), "run paraphrase_generate (and review) first"
    run("bash scripts/06_paraphrase_sweep.sh", log="paraphrase.log", show=IMPORTANT, env={**COMMON, "LLM": LLM})

### results: table, chart and GIFs

In [ ]:
if want("results"):
    from IPython.display import Image, Markdown
    run("bash scripts/07_results.sh", log="results.log")
    display(Markdown(open(OUT / "results" / "results.md").read()))
    display(Image(str(OUT / "results" / "success_rates.png")))

## Finish: final backup (always runs)

In [ ]:
sync_proc.terminate()
run(sync_cmd("prune", "--keep 1"))
run(sync_cmd("push"))
print(f"All backed up to https://huggingface.co/{HF_RUNS_REPO}")

---
## Troubleshooting

- **`uv pip install` fails resolving torch:** remove `--torch-backend=auto` from section 5, or replace it with `--torch-backend=cu126`.
- **Both renderers fail:** run `!apt-get install -y libosmesa6-dev libgl1-mesa-dri` and re-run section 6.
- **Errors mentioning `torchcodec` or video decoding:** set `EXTRA_TRAIN_ARGS = "--dataset.video_backend=pyav"`.
- **CUDA out of memory:** lower `BS` (e.g. 4). Experiment B needs more memory than A; if B only fits at a smaller batch, note it in your write-up.
- **bfloat16 errors:** the T4 has no native bf16. Look for a dtype option with `!lerobot-train --policy.type=smolvla --help | grep -i dtype` and set it to float32 via `EXTRA_TRAIN_ARGS`.
- **A commit run seems stuck:** open *Versions → Logs*. If it's waiting at the LIBERO import, the `printf 'N'` answer in section 5 didn't match the question; run section 5 interactively once to see what LIBERO asks.
- **`--resume` rejected:** check `!lerobot-train --help | grep -i resume` for the current option.

**Tip:** the T4 x2 accelerator gives you two GPUs. In an interactive session you can run an evaluation on the second GPU while training on the first: start the eval from a terminal with `CUDA_VISIBLE_DEVICES=1`.